In [1]:
"""
nfl_odds_model.py

End-to-end NFL "fair odds" model using ESPN public endpoints.

What this script does:
1) Pulls games for the last N days from ESPN scoreboard (date-by-date, ESPN-safe).
2) For each game, pulls ESPN summary/boxscore and extracts TEAM STATS into numeric columns.
3) Builds a team-game table (one row per team per game).
4) Builds matchup rows (HOME vs AWAY), creates rolling pre-game features, then diff features (HOME - AWAY).
5) Trains a model (LogReg baseline by default) + calibrates probabilities.
6) Prints:
   - model performance (accuracy, logloss)
   - latest week of COMPLETED games with your fair odds
   - upcoming games (current week) with predicted fair odds (if enough history)

Dependencies:
  pip install requests pandas numpy scikit-learn
"""

from __future__ import annotations

import time
import math
import requests
import numpy as np
import pandas as pd

from dataclasses import dataclass
from datetime import datetime, timedelta, timezone
from typing import Dict, Any, List, Optional, Tuple

from sklearn.impute import SimpleImputer
from sklearn.metrics import log_loss, accuracy_score
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
# You can try other sklearn models later:
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.ensemble import HistGradientBoostingClassifier


# -----------------------------
# Config
# -----------------------------
ESPN_BASE = "https://site.web.api.espn.com/apis/site/v2/sports/football/nfl"

DEFAULT_N_DAYS_HISTORY = 240   # bigger = more training data
ROLL_WINDOW = 5                # rolling games window (pre-game)
SLEEP_BETWEEN_CALLS = 0.35     # ESPN rate limiting friendliness

# Feature selection
MIN_NON_NULL_RATE = 0.25       # keep features present in >= 25% of rows (change to 0.0 to keep everything)
DROP_CONSTANT_FEATURES = True


# -----------------------------
# Odds helpers
# -----------------------------
def to_american_odds(p: float) -> float:
    """Convert win probability p to fair American odds (no vig)."""
    p = float(p)
    p = min(max(p, 1e-6), 1 - 1e-6)
    if p >= 0.5:
        return -100.0 * p / (1.0 - p)
    return 100.0 * (1.0 - p) / p


def safe_float(x: Any) -> float:
    """Convert ESPN-ish values to float when possible; else NaN."""
    if x is None:
        return np.nan
    if isinstance(x, (int, float, np.number)):
        return float(x)
    if isinstance(x, str):
        s = x.strip().replace("%", "")
        if s == "":
            return np.nan
        # handle "12-34" or "10/20" style -> not numeric; return NaN
        if any(ch in s for ch in ["-", "/"]) and not s.replace("-", "").replace("/", "").replace(".", "").isdigit():
            return np.nan
        try:
            return float(s)
        except Exception:
            return np.nan
    if isinstance(x, dict):
        # common ESPN keys: value, displayValue, stat, amount
        for k in ("value", "displayValue", "stat", "amount"):
            if k in x:
                return safe_float(x[k])
        return np.nan
    return np.nan


# -----------------------------
# ESPN fetching
# -----------------------------
@dataclass
class ESPNClient:
    sleep: float = SLEEP_BETWEEN_CALLS

    def __post_init__(self):
        self.session = requests.Session()
        self.session.headers.update(
            {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)"}
        )

    def get_scoreboard(self, date_yyyymmdd: Optional[str] = None) -> Optional[Dict[str, Any]]:
        url = f"{ESPN_BASE}/scoreboard"
        if date_yyyymmdd:
            url += f"?dates={date_yyyymmdd}"
        try:
            r = self.session.get(url, timeout=20)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            print(f"[scoreboard] error for {date_yyyymmdd}: {e}")
            return None

    def get_summary(self, event_id: str) -> Optional[Dict[str, Any]]:
        url = f"{ESPN_BASE}/summary?event={event_id}"
        try:
            r = self.session.get(url, timeout=25)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            print(f"[summary] error for event={event_id}: {e}")
            return None


def daterange_yyyymmdd(days_back: int) -> List[str]:
    """Return list of YYYYMMDD strings for today back N-1 days."""
    today_utc = datetime.now(timezone.utc).date()
    return [(today_utc - timedelta(days=i)).strftime("%Y%m%d") for i in range(days_back)]


def collect_events_by_dates(client: ESPNClient, days_back: int) -> List[Dict[str, Any]]:
    """Collect raw 'event' objects from ESPN scoreboard, date by date."""
    dates = daterange_yyyymmdd(days_back)
    events: List[Dict[str, Any]] = []

    for ds in dates:
        data = client.get_scoreboard(ds)
        if data and "events" in data:
            events.extend(data["events"])
        time.sleep(client.sleep)

    # de-dup by id
    seen = set()
    uniq = []
    for e in events:
        eid = str(e.get("id"))
        if not eid or eid in seen:
            continue
        seen.add(eid)
        uniq.append(e)
    return uniq


def parse_scoreboard_event_minimal(event_obj: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    """
    Pull teams/home-away + date + completion flag from scoreboard event.
    This is used both for training (completed games) and upcoming (not completed).
    """
    try:
        eid = str(event_obj["id"])
        date_str = event_obj.get("date")
        game_dt = pd.to_datetime(date_str).tz_convert(None) if date_str else pd.NaT

        competition = event_obj["competitions"][0]
        status = competition["status"]["type"]
        completed = bool(status.get("completed", False))

        competitors = competition["competitors"]
        if len(competitors) != 2:
            return None

        teams = []
        for c in competitors:
            team = c["team"]
            teams.append(
                {
                    "event_id": eid,
                    "game_date": game_dt,
                    "team_id": str(team.get("id")),
                    "team": team.get("displayName"),
                    "abbr": team.get("abbreviation"),
                    "home_away": c.get("homeAway"),
                    "score": safe_float(c.get("score")),
                    "winner": bool(c.get("winner", False)),
                    "completed": completed,
                }
            )
        return {"event_id": eid, "game_date": game_dt, "completed": completed, "teams": teams}
    except Exception:
        return None


def extract_team_stats_from_summary(summary_json: Dict[str, Any]) -> Dict[str, Dict[str, float]]:
    """
    Returns dict:
      team_id -> { stat_name: value, ... }

    Attempts to read summary["boxscore"]["teams"][i]["statistics"].
    """
    out: Dict[str, Dict[str, float]] = {}

    box = summary_json.get("boxscore", {}) if isinstance(summary_json, dict) else {}
    teams = box.get("teams", [])
    if not isinstance(teams, list):
        return out

    for t in teams:
        try:
            team_info = t.get("team", {})
            team_id = str(team_info.get("id"))
            if not team_id:
                continue

            stats_map: Dict[str, float] = {}

            stats_list = t.get("statistics", [])
            if isinstance(stats_list, list):
                for s in stats_list:
                    # ESPN often has: name, abbreviation, displayValue, value
                    name = s.get("name") or s.get("abbreviation")
                    if not name:
                        continue
                    val = s.get("value")
                    if val is None:
                        val = s.get("displayValue")
                    stats_map[name] = safe_float(val)

            out[team_id] = stats_map
        except Exception:
            continue

    return out


def build_team_game_table(
    client: ESPNClient,
    events: List[Dict[str, Any]],
    keep_incomplete: bool = False,
) -> pd.DataFrame:
    """
    Build one row per TEAM per GAME with:
      event_id, game_date, team_id, team, abbr, home_away,
      points, opp_points, plus many numeric team boxscore stats.

    If keep_incomplete=False, only keeps completed games w/ scores.
    """
    rows: List[Dict[str, Any]] = []

    for ev in events:
        minimal = parse_scoreboard_event_minimal(ev)
        if not minimal:
            continue

        eid = minimal["event_id"]
        game_date = minimal["game_date"]
        completed = minimal["completed"]

        # If not keeping incomplete and not completed => skip
        if (not keep_incomplete) and (not completed):
            continue

        # Pull summary for richer stats (works for completed; may be empty for upcoming)
        summary = client.get_summary(eid) if completed else None
        team_stats = extract_team_stats_from_summary(summary) if summary else {}

        teams = minimal["teams"]
        if len(teams) != 2:
            continue

        # identify opponents
        a, b = teams[0], teams[1]
        by_id = {a["team_id"]: a, b["team_id"]: b}

        for tm in teams:
            points = tm["score"]
            opp = b if tm["team_id"] == a["team_id"] else a
            opp_points = opp["score"]

            # if completed, require scores
            if completed and (pd.isna(points) or pd.isna(opp_points)):
                continue

            r = {
                "event_id": str(eid),
                "game_date": game_date,
                "team_id": tm["team_id"],
                "team": tm["team"],
                "abbr": tm["abbr"],
                "home_away": tm["home_away"],
                "completed": completed,
                "points": points,
                "opp_points": opp_points,
            }

            # merge in team stats (may be empty for upcoming)
            stats = team_stats.get(tm["team_id"], {})
            for k, v in stats.items():
                # prefix to avoid collisions with core cols
                r[f"stat_{k}"] = v

            rows.append(r)

        time.sleep(client.sleep)

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    # Clean types
    df["game_date"] = pd.to_datetime(df["game_date"], errors="coerce")
    df["event_id"] = df["event_id"].astype(str)
    df["team_id"] = df["team_id"].astype(str)

    # Drop duplicate columns just in case
    df = df.loc[:, ~df.columns.duplicated()].copy()

    # Ensure numeric for stat_*
    for c in df.columns:
        if c.startswith("stat_") or c in ("points", "opp_points"):
            df[c] = pd.to_numeric(df[c], errors="coerce")

    return df


# -----------------------------
# Dataset building (matchups + rolling)
# -----------------------------
def make_matchup_dataset(team_game: pd.DataFrame, window: int = ROLL_WINDOW) -> Tuple[pd.DataFrame, List[str]]:
    """
    Builds home-vs-away matchup rows from team_game, adds rolling pre-game features,
    then constructs diff features (home - away).

    Returns: (df_matchups, feature_cols)
    """
    tg = team_game.copy()
    tg = tg.dropna(subset=["event_id", "team_id", "game_date"]).copy()
    tg = tg.sort_values(["team_id", "game_date", "event_id"]).reset_index(drop=True)

    # rolling features for ALL numeric stats except identifiers/leakage
    numeric_cols = [c for c in tg.columns if pd.api.types.is_numeric_dtype(tg[c])]
    # exclude obvious leakage-ish columns (points are OK if shifted, but you might drop it later)
    exclude = {"points", "opp_points"}
    numeric_cols = [c for c in numeric_cols if c not in exclude]

    for c in numeric_cols + ["points", "opp_points"]:
        # shifted prevents leakage (uses only games BEFORE current)
        tg[f"roll_{c}_l{window}"] = (
            tg.groupby("team_id")[c]
              .shift(1)
              .rolling(window)
              .mean()
        )

    # Build matchup rows using HOME/AWAY from the same event_id
    base_cols = ["event_id", "game_date", "team_id", "team", "abbr", "home_away", "points"]
    g = tg[base_cols].copy()
    g = g.sort_values(["event_id", "home_away"])

    pairs = []
    for eid, grp in g.groupby("event_id"):
        if len(grp) != 2:
            continue
        # pick home/away explicitly
        home = grp[grp["home_away"] == "home"]
        away = grp[grp["home_away"] == "away"]
        if len(home) != 1 or len(away) != 1:
            continue
        home = home.iloc[0]
        away = away.iloc[0]

        pairs.append(
            {
                "event_id": str(eid),
                "game_date": home["game_date"],
                "home_team": home["team"],
                "away_team": away["team"],
                "home_id": home["team_id"],
                "away_id": away["team_id"],
                "pts_home": home["points"],
                "pts_away": away["points"],
                "y_home_win": int(home["points"] > away["points"]) if (not pd.isna(home["points"]) and not pd.isna(away["points"])) else np.nan,
            }
        )

    games = pd.DataFrame(pairs)
    if games.empty:
        return games, []

    # Feature columns = all rolling cols
    roll_cols = [c for c in tg.columns if c.startswith("roll_")]

    feats = tg[["event_id", "team_id"] + roll_cols].copy()

    # join rolling features for home + away
    df = games.merge(feats, left_on=["event_id", "home_id"], right_on=["event_id", "team_id"], how="left")
    df = df.drop(columns=["team_id"]).rename(columns={c: f"h_{c}" for c in roll_cols})

    df = df.merge(feats, left_on=["event_id", "away_id"], right_on=["event_id", "team_id"], how="left")
    df = df.drop(columns=["team_id"]).rename(columns={c: f"a_{c}" for c in roll_cols})

    # diff features: home - away
    feature_cols: List[str] = []
    for c in roll_cols:
        hc = f"h_{c}"
        ac = f"a_{c}"
        dc = f"d_{c}"
        if hc in df.columns and ac in df.columns:
            df[dc] = df[hc] - df[ac]
            feature_cols.append(dc)

    # sort + keep only rows with label
    df["game_date"] = pd.to_datetime(df["game_date"], errors="coerce")
    df = df.dropna(subset=["game_date"]).sort_values("game_date").reset_index(drop=True)

    # y is only for completed games; keep it for training rows
    return df, feature_cols


# -----------------------------
# Training + calibration
# -----------------------------
def train_calibrated_model(df: pd.DataFrame, features: List[str]) -> Tuple[Any, SimpleImputer, pd.DataFrame, np.ndarray, np.ndarray, np.ndarray]:
    """
    70/15/15 time split:
      train -> fit base model
      cal   -> fit calibration
      test  -> evaluate

    Returns: (calibrated_model, imputer, test_df, y_test, p_test, FEATURES_USED)
    """
    df = df.copy()
    df = df.dropna(subset=["y_home_win"]).copy()
    df["y_home_win"] = df["y_home_win"].astype(int)

    # ensure numeric
    for c in features:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # drop all-NaN features and (optional) constants
    feats = [c for c in features if df[c].notna().any()]
    if DROP_CONSTANT_FEATURES:
        feats = [c for c in feats if df[c].nunique(dropna=True) > 1]

    # missingness filter (lower it if you want more features)
    if MIN_NON_NULL_RATE > 0:
        coverage = df[feats].notna().mean()
        feats = coverage[coverage >= MIN_NON_NULL_RATE].index.tolist()

    print(f"Features available: {len(features)}")
    print(f"Features used after cleanup/filter: {len(feats)}")
    if len(feats) < 5:
        print("WARNING: very few features survived. That usually means ESPN stats are not being extracted well yet.")

    df = df.sort_values("game_date").reset_index(drop=True)
    n = len(df)
    cut_train = int(n * 0.70)
    cut_cal = int(n * 0.85)

    train_df = df.iloc[:cut_train]
    cal_df = df.iloc[cut_train:cut_cal]
    test_df = df.iloc[cut_cal:]

    imp = SimpleImputer(strategy="median")
    X_train = imp.fit_transform(train_df[feats])
    y_train = train_df["y_home_win"].values

    X_cal = imp.transform(cal_df[feats])
    y_cal = cal_df["y_home_win"].values

    X_test = imp.transform(test_df[feats])
    y_test = test_df["y_home_win"].values

    # Baseline model (good starting point)
    base = LogisticRegression(
        solver="liblinear",
        max_iter=6000,
        C=0.7,  # lower = more regularization
    )
    base.fit(X_train, y_train)

    # Calibration (makes odds less insane)
    cal = CalibratedClassifierCV(base, method="sigmoid", cv="prefit")
    cal.fit(X_cal, y_cal)

    p_test = cal.predict_proba(X_test)[:, 1]

    y_hat = (p_test >= 0.5).astype(int)
    print("Test accuracy:", round(accuracy_score(y_test, y_hat), 4))
    print("Test logloss: ", round(log_loss(y_test, p_test), 4))

    return cal, imp, test_df, y_test, p_test, np.array(feats)


# -----------------------------
# Upcoming prediction helper
# -----------------------------
def predict_upcoming_week(
    client: ESPNClient,
    team_game: pd.DataFrame,
    model: Any,
    imputer: SimpleImputer,
    feature_cols: List[str],
    window: int,
    features_used: np.ndarray,
) -> pd.DataFrame:
    """
    Predict fair odds for current-week (upcoming/incomplete) games.
    We:
      - pull current scoreboard (no date -> current week)
      - build minimal matchup list home/away
      - compute last rolling features per team from historical team_game
      - join and predict
    """
    sb = client.get_scoreboard(date_yyyymmdd=None)
    if not sb or "events" not in sb:
        return pd.DataFrame()

    events = sb["events"]
    mini = []
    for ev in events:
        parsed = parse_scoreboard_event_minimal(ev)
        if not parsed or parsed["completed"]:
            continue
        # need 2 teams
        teams = parsed["teams"]
        if len(teams) != 2:
            continue
        home = [t for t in teams if t["home_away"] == "home"]
        away = [t for t in teams if t["home_away"] == "away"]
        if len(home) != 1 or len(away) != 1:
            continue
        home, away = home[0], away[0]

        mini.append(
            {
                "event_id": parsed["event_id"],
                "game_date": parsed["game_date"],
                "home_team": home["team"],
                "away_team": away["team"],
                "home_id": home["team_id"],
                "away_id": away["team_id"],
            }
        )

    upcoming = pd.DataFrame(mini)
    if upcoming.empty:
        return upcoming

    # Build rolling features from historical team_game the same way
    tg = team_game.copy()
    tg = tg.dropna(subset=["team_id", "game_date"]).sort_values(["team_id", "game_date", "event_id"]).reset_index(drop=True)

    # numeric columns
    numeric_cols = [c for c in tg.columns if pd.api.types.is_numeric_dtype(tg[c])]
    exclude = {"points", "opp_points"}
    numeric_cols = [c for c in numeric_cols if c not in exclude]
    for c in numeric_cols + ["points", "opp_points"]:
        tg[f"roll_{c}_l{window}"] = tg.groupby("team_id")[c].shift(1).rolling(window).mean()

    roll_cols = [c for c in tg.columns if c.startswith("roll_")]

    # take most recent rolling row per team
    last = tg.sort_values("game_date").groupby("team_id").tail(1)[["team_id"] + roll_cols].copy()

    # join for home and away
    df = upcoming.merge(last, left_on="home_id", right_on="team_id", how="left").drop(columns=["team_id"])
    df = df.rename(columns={c: f"h_{c}" for c in roll_cols})

    df = df.merge(last, left_on="away_id", right_on="team_id", how="left").drop(columns=["team_id"])
    df = df.rename(columns={c: f"a_{c}" for c in roll_cols})

    # diff features
    for c in roll_cols:
        df[f"d_{c}"] = df.get(f"h_{c}") - df.get(f"a_{c}")

    # only keep the exact features the model was trained on
    for c in features_used:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    X = imputer.transform(df[features_used])
    p = model.predict_proba(X)[:, 1]

    df["p_home_win"] = p
    df["fair_american_home"] = df["p_home_win"].apply(to_american_odds).round(1)
    df["fair_american_away"] = (1 - df["p_home_win"]).apply(to_american_odds).round(1)

    return df.sort_values("game_date")


# -----------------------------
# Main
# -----------------------------
def main():
    client = ESPNClient(sleep=SLEEP_BETWEEN_CALLS)

    print("\n=== 1) Collect historical events ===")
    events_hist = collect_events_by_dates(client, days_back=DEFAULT_N_DAYS_HISTORY)
    print(f"Events collected (raw): {len(events_hist)}")

    print("\n=== 2) Build team-game table (completed games) ===")
    team_game = build_team_game_table(client, events_hist, keep_incomplete=False)
    if team_game.empty:
        print("No completed games found. Exiting.")
        return

    print("team_game shape:", team_game.shape)
    print("team_game date range:", team_game["game_date"].min(), "→", team_game["game_date"].max())

    num_cols = team_game.select_dtypes(include="number").columns.tolist()
    print("Numeric cols:", len(num_cols))
    print("Example numeric cols:", num_cols[:30])

    print("\n=== 3) Build matchup dataset + rolling features ===")
    df, FEATURE_COLS = make_matchup_dataset(team_game, window=ROLL_WINDOW)
    if df.empty or not FEATURE_COLS:
        print("Could not build matchup dataset / no features. Exiting.")
        return

    print("Matchups:", len(df))
    print("Feature candidates:", len(FEATURE_COLS))
    print("Feature example:", FEATURE_COLS[:10])

    print("\n=== 4) Train + calibrate model ===")
    model, imputer, test_df, y_test, p_test, FEATURES_USED = train_calibrated_model(df, FEATURE_COLS)

    # Completed games odds table (latest week in test_df)
    out = test_df[["game_date", "home_team", "away_team", "pts_home", "pts_away", "y_home_win"]].copy()
    out["p_home_win"] = np.round(p_test, 3)
    out["fair_american_home"] = out["p_home_win"].apply(to_american_odds).round(1)
    out["fair_american_away"] = (1 - out["p_home_win"]).apply(to_american_odds).round(1)

    latest_day = out["game_date"].max()
    week_start = latest_day - pd.Timedelta(days=7)

    print("\n=== LATEST WEEK (COMPLETED, from TEST SPLIT) ===")
    print(
        out[out["game_date"] >= week_start]
        .sort_values("game_date", ascending=False)
        .head(30)
        .to_string(index=False)
    )

    print("\n=== 5) Predict current-week UPCOMING games ===")
    upcoming = predict_upcoming_week(
        client=client,
        team_game=team_game,
        model=model,
        imputer=imputer,
        feature_cols=FEATURE_COLS,
        window=ROLL_WINDOW,
        features_used=FEATURES_USED,
    )

    if upcoming.empty:
        print("No upcoming games found (or ESPN scoreboard unavailable).")
    else:
        cols = ["game_date", "home_team", "away_team", "p_home_win", "fair_american_home", "fair_american_away"]
        print(upcoming[cols].head(30).to_string(index=False))

    print("\n=== Done ===")
    print("Features used:", len(FEATURES_USED))
    print("First 25:", FEATURES_USED[:25])


if __name__ == "__main__":
    main()



=== 1) Collect historical events ===
Events collected (raw): 294

=== 2) Build team-game table (completed games) ===
team_game shape: (584, 33)
team_game date range: 2025-08-01 00:00:00 → 2025-12-26 01:15:00
Numeric cols: 26
Example numeric cols: ['points', 'opp_points', 'stat_firstDowns', 'stat_firstDownsPassing', 'stat_firstDownsRushing', 'stat_firstDownsPenalty', 'stat_thirdDownEff', 'stat_fourthDownEff', 'stat_totalOffensivePlays', 'stat_totalYards', 'stat_yardsPerPlay', 'stat_totalDrives', 'stat_netPassingYards', 'stat_completionAttempts', 'stat_yardsPerPass', 'stat_interceptions', 'stat_sacksYardsLost', 'stat_rushingYards', 'stat_rushingAttempts', 'stat_yardsPerRushAttempt', 'stat_redZoneAttempts', 'stat_totalPenaltiesYards', 'stat_turnovers', 'stat_fumblesLost', 'stat_defensiveTouchdowns', 'stat_possessionTime']

=== 3) Build matchup dataset + rolling features ===
Matchups: 292
Feature candidates: 27
Feature example: ['d_roll_completed_l5', 'd_roll_stat_firstDowns_l5', 'd_roll_

/Users/rayanarya/Library/Python/3.12/lib/python/site-packages/sklearn/calibration.py:330: FutureWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(


          game_date            home_team            away_team  p_home_win  fair_american_home  fair_american_away
2025-12-27 21:30:00 Los Angeles Chargers       Houston Texans    0.370355               170.0              -170.0
2025-12-28 01:00:00    Green Bay Packers     Baltimore Ravens    0.548884              -121.7               121.7
2025-12-28 18:00:00   Cincinnati Bengals    Arizona Cardinals    0.425741               134.9              -134.9
2025-12-28 18:00:00     Cleveland Browns  Pittsburgh Steelers    0.532039              -113.7               113.7
2025-12-28 18:00:00     Tennessee Titans   New Orleans Saints    0.323759               208.9              -208.9
2025-12-28 18:00:00   Indianapolis Colts Jacksonville Jaguars    0.387454               158.1              -158.1
2025-12-28 18:00:00       Miami Dolphins Tampa Bay Buccaneers    0.871815              -680.1               680.1
2025-12-28 18:00:00        New York Jets New England Patriots    0.282383               